## Similarity search

In this notebook we will perform similarity search on the selected Deep Drilling Fields (ECDFS and EDFS).

In [4]:
import lsdb
import matplotlib.pyplot as plt
import nested_pandas as npd
import numpy as np

Let's select the targets to query for:

In [5]:
targets = npd.read_parquet("outputs/target_embeddings.parquet")

Let's also load the ECDFS+EDFS embeddings:

In [7]:
ddf_embeddings = lsdb.open_catalog("outputs/ddf_embeddings")

#### Note

Finding the N-most similar objects is not trivial in a distributed computation.

Let's define a method that can run on `map_partitions` and get the top-5 similarity samples for each query:
- When we run the similarity search the result will only contain up to K results per partition. 
- We will have up to `5 rows * 33_850 partitions` (for DP2) which is very manageable.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

top_k = 5

def top_k_similar(df, query_embedding):
    """Pick up to top_k rows from df that are most similar to query_embedding."""
    if len(df) == 0:
        return df.assign(top_scores=[])
    # The embeddings have 384 features
    embeddings = np.asarray(df["embeddings.value"]).reshape(-1,384)
    scores = cosine_similarity(query_embedding, embeddings)
    top_indices = np.argsort(scores, axis=1)[:, ::-1][:, :top_k]
    top_scores = np.take_along_axis(scores, top_indices, axis=1)
    # Select only the top_k rows and add the top_scores as a new column.
    return df.iloc[top_indices.flatten()].assign(top_scores=top_scores.flatten())

def query_best_matches(target_row):
    query = np.asarray(target_row.embeddings.value).reshape(1, -1)
    return (
        ddf_embeddings
        .map_partitions(top_k_similar, query_embedding=query)
        .compute()
        .nlargest(top_k, "top_scores")
    )

Now use the First Look target embeddings as "queries":

In [9]:
from lsdb_rubin.plot_light_curve import plot_light_curve

def plot_matches(result, axes):
    for j, match in enumerate(result.itertuples()):
        plt.sca(axes[j])
        plot_light_curve(match.diaSource, flux_field="psfFlux")
        axes[j].set_title(f"diaObjectId={match.diaObjectId}\nscore={match.top_scores:.4f}", fontsize=9)

for target in targets.itertuples():
    fig, axes = plt.subplots(1, top_k, figsize=(4 * top_k, 4))
    axes = axes.flatten()
    fig.suptitle(f"Best matches for target diaObjectId={target.diaObjectId_dp2}", fontsize=14)
    result = query_best_matches(target)
    plot_matches(result, axes)
    plt.tight_layout()
    plt.show()

In the next notebook we will try to generate the embeddings for the whole filtered DP2.